# 🛡️ SafeGuard Vision AI - Video Color Extractor

**Proyecto:** MIT Global Teaching Labs  
**Autor:** Grupo Outliers

Este notebook extrae la parte a **COLOR** de videos que tienen formato dividido (binario + color).

---

## 📌 Paso 1: Montar Google Drive

Ejecuta esta celda y autoriza el acceso a tu Drive.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

print("\n✅ Google Drive montado correctamente!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

✅ Google Drive montado correctamente!


## 📌 Paso 2: Configurar las rutas

⚠️ **IMPORTANTE:** Modifica las rutas según tu estructura de carpetas en Drive.

**Ejemplo:** Si tu carpeta está en `Mi unidad > Proyecto > Videos`, la ruta sería:
```
/content/drive/MyDrive/Proyecto/Videos
```

In [8]:
# ============================================================
# 🔧 CONFIGURA ESTAS RUTAS (modifica según tu Drive)
# ============================================================

# Carpeta donde están tus videos originales en Drive
INPUT_FOLDER = "/content/drive/MyDrive/ur_fall/adl"

# Carpeta donde se guardarán los videos procesados (solo color)
OUTPUT_FOLDER = "/content/drive/MyDrive/ur_fall/adl_color"

# ============================================================

import os

# Verificar que la carpeta de entrada existe
if os.path.exists(INPUT_FOLDER):
    # Filtrar SOLO videos con "cam0" (vista frontal)
    all_videos = [f for f in os.listdir(INPUT_FOLDER) if f.lower().endswith(('.mp4', '.avi', '.mov', '.mkv'))]
    videos = [f for f in all_videos if 'cam0' in f.lower()]

    print(f"✅ Carpeta de entrada encontrada: {INPUT_FOLDER}")
    print(f"   📹 Videos totales: {len(all_videos)}")
    print(f"   🎯 Videos cam0 (frontal): {len(videos)} ← SOLO ESTOS SE PROCESARÁN")
    print(f"   ⏭️  Videos cam1 (ignorados): {len(all_videos) - len(videos)}")
    if videos:
        print(f"\n   Primeros archivos cam0: {videos[:5]}")
else:
    print(f"❌ ERROR: La carpeta no existe: {INPUT_FOLDER}")
    print("\n💡 Verifica la ruta. Puedes explorar tu Drive en el panel izquierdo.")

# Crear carpeta de salida si no existe
os.makedirs(OUTPUT_FOLDER, exist_ok=True)
print(f"\n✅ Carpeta de salida: {OUTPUT_FOLDER}")

✅ Carpeta de entrada encontrada: /content/drive/MyDrive/ur_fall/adl
   📹 Videos totales: 40
   🎯 Videos cam0 (frontal): 40 ← SOLO ESTOS SE PROCESARÁN
   ⏭️  Videos cam1 (ignorados): 0

   Primeros archivos cam0: ['adl-06-cam0.mp4', 'adl-04-cam0.mp4', 'adl-20-cam0.mp4', 'adl-18-cam0.mp4', 'adl-38-cam0.mp4']

✅ Carpeta de salida: /content/drive/MyDrive/ur_fall/adl_color


## 📌 Paso 3: Explorar tu Drive (opcional)

Si no estás seguro de la ruta, ejecuta esta celda para ver la estructura de tu Drive.

In [ ]:
# Explorar carpetas en la raíz de tu Drive
drive_root = "/content/drive/MyDrive"

print("📂 Carpetas en la raíz de tu Drive:")
print("-" * 40)

for item in sorted(os.listdir(drive_root))[:20]:  # Mostrar primeras 20
    full_path = os.path.join(drive_root, item)
    if os.path.isdir(full_path):
        print(f"📁 {item}")
    else:
        print(f"📄 {item}")

print("\n💡 Usa estas rutas para configurar INPUT_FOLDER arriba.")

## 📌 Paso 4: Definir funciones de procesamiento

In [9]:
import cv2
import numpy as np
from pathlib import Path
from tqdm.notebook import tqdm
import time

def remove_black_bars(frame):
    """
    Detecta y remueve las barras negras (letterbox) del frame.
    """
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Encontrar filas que NO son completamente negras
    row_means = gray.mean(axis=1)
    threshold = 10  # Píxeles con valor < 10 se consideran negros

    non_black_rows = row_means > threshold

    # Encontrar inicio y fin del contenido
    if non_black_rows.any():
        first_row = non_black_rows.argmax()
        last_row = len(non_black_rows) - non_black_rows[::-1].argmax()
        return frame[first_row:last_row, :]

    return frame


def extract_right_half(frame):
    """
    Extrae la mitad derecha del frame (donde está el video a color).
    """
    height, width = frame.shape[:2]
    return frame[:, width//2:]


def process_video(input_path, output_path):
    """
    Procesa un video: remueve barras negras y extrae la parte a color.
    """
    cap = cv2.VideoCapture(str(input_path))

    if not cap.isOpened():
        return False, "No se pudo abrir el video"

    # Obtener propiedades
    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps <= 0:
        fps = 30  # Default
    fps = int(fps)

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    # Leer primer frame para calcular dimensiones finales
    ret, first_frame = cap.read()
    if not ret:
        cap.release()
        return False, "No se pudo leer el primer frame"

    # Procesar primer frame para obtener dimensiones
    no_bars = remove_black_bars(first_frame)
    color_region = extract_right_half(no_bars)

    final_height, final_width = color_region.shape[:2]

    # Configurar escritor de video
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(str(output_path), fourcc, fps, (final_width, final_height))

    # Volver al inicio
    cap.set(cv2.CAP_PROP_POS_FRAMES, 0)

    # Procesar todos los frames
    for _ in range(total_frames):
        ret, frame = cap.read()
        if not ret:
            break

        # 1. Remover barras negras
        no_bars = remove_black_bars(frame)

        # 2. Extraer mitad derecha (color)
        color_frame = extract_right_half(no_bars)

        # 3. Escribir al video de salida
        out.write(color_frame)

    cap.release()
    out.release()

    return True, f"{final_width}x{final_height}"


print("✅ Funciones de procesamiento cargadas!")

✅ Funciones de procesamiento cargadas!


## 📌 Paso 5: Vista previa de un video (opcional)

Verifica que el script detecta correctamente la región a color.

In [ ]:
import matplotlib.pyplot as plt

# Obtener primer video para preview
videos = [f for f in os.listdir(INPUT_FOLDER) if f.lower().endswith(('.mp4', '.avi', '.mov', '.mkv'))]

if videos:
    test_video = os.path.join(INPUT_FOLDER, videos[0])
    print(f"📹 Preview de: {videos[0]}")

    cap = cv2.VideoCapture(test_video)
    ret, frame = cap.read()
    cap.release()

    if ret:
        # Procesar frame
        no_bars = remove_black_bars(frame)
        color_only = extract_right_half(no_bars)

        # Mostrar comparación
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))

        axes[0].imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        axes[0].set_title(f"Original\n{frame.shape[1]}x{frame.shape[0]}")
        axes[0].axis('off')

        axes[1].imshow(cv2.cvtColor(no_bars, cv2.COLOR_BGR2RGB))
        axes[1].set_title(f"Sin barras negras\n{no_bars.shape[1]}x{no_bars.shape[0]}")
        axes[1].axis('off')

        axes[2].imshow(cv2.cvtColor(color_only, cv2.COLOR_BGR2RGB))
        axes[2].set_title(f"Solo COLOR (resultado)\n{color_only.shape[1]}x{color_only.shape[0]}")
        axes[2].axis('off')

        plt.tight_layout()
        plt.show()

        print("\n✅ Si el tercer panel muestra solo la parte a COLOR, el script está bien configurado!")
else:
    print("❌ No se encontraron videos para preview")

## 📌 Paso 6: ¡Procesar todos los videos! 🚀

Esta celda procesa **todos** los videos de la carpeta de entrada y los guarda en la carpeta de salida.

In [10]:
from tqdm.notebook import tqdm
import time

print("="*65)
print("🛡️  SafeGuard Vision AI - Video Color Extractor")
print("="*65)

# Buscar videos SOLO cam0
video_extensions = ['.mp4', '.avi', '.mov', '.mkv']
videos = []

for f in os.listdir(INPUT_FOLDER):
    if any(f.lower().endswith(ext) for ext in video_extensions):
        if 'cam0' in f.lower():  # ← FILTRO CAM0
            videos.append(f)

videos = sorted(videos)

if not videos:
    print(f"\n❌ No se encontraron videos cam0 en: {INPUT_FOLDER}")
else:
    print(f"\n📂 Entrada:  {INPUT_FOLDER}")
    print(f"📂 Salida:   {OUTPUT_FOLDER}")
    print(f"🎯 Videos cam0: {len(videos)}")
    print("-"*65)

    successful = 0
    failed = 0
    start_time = time.time()

    for i, video_name in enumerate(videos, 1):
        input_path = os.path.join(INPUT_FOLDER, video_name)
        output_name = f"color_{Path(video_name).stem}.mp4"
        output_path = os.path.join(OUTPUT_FOLDER, output_name)

        print(f"\n[{i}/{len(videos)}] 📹 {video_name}")

        success, info = process_video(input_path, output_path)

        if success:
            print(f"   ✅ Guardado: {output_name} ({info})")
            successful += 1
        else:
            print(f"   ❌ Error: {info}")
            failed += 1

    # Resumen
    elapsed = time.time() - start_time

    print("\n" + "="*65)
    print("📊 RESUMEN")
    print("="*65)
    print(f"   ✅ Exitosos:     {successful}")
    print(f"   ❌ Errores:      {failed}")
    print(f"   ⏱️  Tiempo total: {elapsed:.1f} segundos")
    print(f"   📂 Guardados en: {OUTPUT_FOLDER}")
    print("="*65)
    print("\n🎉 ¡Completado! Los videos ya están en tu Google Drive.")

🛡️  SafeGuard Vision AI - Video Color Extractor

📂 Entrada:  /content/drive/MyDrive/ur_fall/adl
📂 Salida:   /content/drive/MyDrive/ur_fall/adl_color
🎯 Videos cam0: 40
-----------------------------------------------------------------

[1/40] 📹 adl-01-cam0.mp4
   ✅ Guardado: color_adl-01-cam0.mp4 (320x240)

[2/40] 📹 adl-02-cam0.mp4
   ✅ Guardado: color_adl-02-cam0.mp4 (320x240)

[3/40] 📹 adl-03-cam0.mp4
   ✅ Guardado: color_adl-03-cam0.mp4 (320x240)

[4/40] 📹 adl-04-cam0.mp4
   ✅ Guardado: color_adl-04-cam0.mp4 (320x240)

[5/40] 📹 adl-05-cam0.mp4
   ✅ Guardado: color_adl-05-cam0.mp4 (320x240)

[6/40] 📹 adl-06-cam0.mp4
   ✅ Guardado: color_adl-06-cam0.mp4 (320x240)

[7/40] 📹 adl-07-cam0.mp4
   ✅ Guardado: color_adl-07-cam0.mp4 (320x240)

[8/40] 📹 adl-08-cam0.mp4
   ✅ Guardado: color_adl-08-cam0.mp4 (320x240)

[9/40] 📹 adl-09-cam0.mp4
   ✅ Guardado: color_adl-09-cam0.mp4 (320x240)

[10/40] 📹 adl-10-cam0.mp4
   ✅ Guardado: color_adl-10-cam0.mp4 (320x240)

[11/40] 📹 adl-11-cam0.mp4
   ✅ Guar

## 📌 Paso 7: Verificar resultados

Lista los videos procesados en la carpeta de salida.

In [ ]:
print("📂 Videos procesados en la carpeta de salida:")
print("-" * 50)

output_videos = [f for f in os.listdir(OUTPUT_FOLDER) if f.lower().endswith(('.mp4', '.avi', '.mov', '.mkv'))]
output_videos = sorted(output_videos)

for i, video in enumerate(output_videos, 1):
    file_path = os.path.join(OUTPUT_FOLDER, video)
    size_mb = os.path.getsize(file_path) / (1024 * 1024)
    print(f"   {i}. {video} ({size_mb:.1f} MB)")

print(f"\n📊 Total: {len(output_videos)} videos")
print(f"📂 Ubicación: {OUTPUT_FOLDER}")

---

## ✅ ¡Listo!

Los videos procesados ya están en tu Google Drive en la carpeta especificada.

**Próximos pasos para SafeGuard Vision AI:**
1. Usar estos videos para extraer poses con BlazePose/MediaPipe
2. Entrenar el modelo de detección de caídas
3. Evaluar con el dataset

---
*SafeGuard Vision AI - MIT Global Teaching Labs*